# 🌟 Exercise: A local app running MCP (Model Context Protocol)

**Goal:** build a tiny MCP **server** (one tool + one resource) and a Python **client** that connects over **STDIO**, discovers the server's features and invokes them.

### Hosts, clients, servers, and why STDIO
- **Host:** the application the user interacts with (Claude Desktop, an IDE, a chatbot). It decides when to use external context.
- **Client:** lives inside the host, holds a **1:1 connection** with one server, and speaks the MCP JSON-RPC protocol (`initialize`, `tools/list`, `tools/call`, `resources/list`, `resources/read`…).
- **Server:** exposes capabilities:
  - **tools**: *actions* the model can call, with JSON-schema arguments;
  - **resources**: *read-only context* addressed by URIs;
  - **prompts**.
- **STDIO transport:** the client **spawns the server as a subprocess** and exchanges JSON-RPC messages over its stdin/stdout. It needs no network, ports, authentication or deployment, and the server's lifetime is tied to the client. That is ideal for local development. HTTP/SSE transports are for remote, multi-client servers.

This notebook runs as-is in Colab: the `%%writefile` cells create `server.py` and `client.py`, and `!python client.py` plays the role of the terminal.

## Setup

In [1]:
# The course command is: pip install "mcp[cli]"
# Since mcp 2.x, FastMCP was renamed (MCPServer) and the scaffold below fails with
# "No module named 'mcp.server.fastmcp'" -> pin the v1 SDK that the exercise is written for.
%pip install -q "mcp[cli]<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
!python --version
!python -c "from importlib.metadata import version; print('mcp', version('mcp'))"
!mcp --help | head -20

Python 3.13.16


mcp 1.30.0


                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                


 MCP development tools                                                          
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ version  Show the MCP version.                                               │
│ dev      Run an MCP server with the MCP Inspector.                           │
│ run      Run an MCP server.                                                  │
│ install  Install an MCP server in the Claude desktop app.                    │
╰──────────────────────────────────────────────────────────────────────────────╯



## A. Server (`server.py`)
- `FastMCP("Demo")` creates a server named **Demo**.
- `@mcp.tool()` registers `add(a: int, b: int) -> int`. FastMCP derives the JSON input schema from the type hints and uses the docstring as the description.
- `@mcp.resource("greeting://{name}")` registers a **resource template**: `{name}` in the URI is passed to the function as an argument.
- `mcp.run(transport="stdio")` starts the server loop on stdin/stdout.

In [3]:
%%writefile server.py
# server.py - a tiny MCP server exposing one tool and one resource over STDIO
from mcp.server.fastmcp import FastMCP

# Create an MCP server with a name
mcp = FastMCP("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a personalised greeting (read-only context)."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    # Start the server loop on STDIO (default transport)
    mcp.run(transport="stdio")

Overwriting server.py


## B. Client (`client.py`)
- `StdioServerParameters(command="mcp", args=["run", "server.py"])` tells the client how to **spawn** the server.
- `stdio_client(...)` opens the subprocess and returns its read/write streams. `ClientSession` speaks MCP over them.
- `initialize()` performs the protocol handshake (versions and capabilities).
- We list **resources** and **resource templates**. `greeting://{name}` is a *template*, so it appears in `list_resource_templates()`, while `list_resources()` only returns concrete, static resources. Then we list **tools**, read `greeting://hello` and call `add` with `a=1, b=7`.

In [4]:
%%writefile client.py
# client.py - spawns server.py over STDIO, discovers its features and invokes them
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            print(f"Connected to server: {init.serverInfo.name} (protocol {init.protocolVersion})")

            # List resources: concrete resources and URI templates such as greeting://{name}
            resources = await session.list_resources()
            templates = await session.list_resource_templates()
            print("Resources:", [str(r.uri) for r in resources.resources] or "(no static resources)")
            print("Resource templates:", [t.uriTemplate for t in templates.resourceTemplates])

            # List tools
            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])
            for t in tools.tools:
                print(f"  - {t.name}: {t.description} | input schema: {t.inputSchema}")

            # Read greeting://hello
            content = await session.read_resource("greeting://hello")
            print("greeting://hello ->", content.contents[0].text)

            # Call the add tool with a=1, b=7
            result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print("add(1, 7) ->", result.content[0].text,
                  "| structured:", getattr(result, "structuredContent", None))


if __name__ == "__main__":
    asyncio.run(run())

Overwriting client.py


## C. Run
### Option 1: one terminal (the client spawns the server)

In [5]:
# The server's INFO logs go to stderr; 2>&1 shows the full terminal capture
!python client.py 2>&1 | tee terminal_capture.txt

Connected to server: Demo (protocol 2025-11-25)
[10/12/26 12:30:13] INFO     Processing request of type            server.py:733
                             ListResourcesRequest                               
                    INFO     Processing request of type            server.py:733
                             ListResourceTemplatesRequest                       
Resources: (no static resources)
Resource templates: ['greeting://{name}']
                    INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
Tools: ['add']
  - add: Add two integers and return the sum. | input schema: {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'}


                    INFO     Processing request of type            server.py:733
                             ReadResourceRequest                                
greeting://hello -> Hello, hello!
                    INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
add(1, 7) -> 8 | structured: {'result': 8}


In [6]:
# Clean view (stdout only): the four things to submit
!python client.py 2>/dev/null

Connected to server: Demo (protocol 2025-11-25)
Resources: (no static resources)
Resource templates: ['greeting://{name}']
Tools: ['add']
  - add: Add two integers and return the sum. | input schema: {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'}


greeting://hello -> Hello, hello!
add(1, 7) -> 8 | structured: {'result': 8}


### Option 2: check the server on its own (`mcp run server.py`)
In a second terminal you would run `mcp run server.py`. The server then waits for JSON-RPC messages on stdin. Here we start it for a few seconds to check that it **boots without errors**, which is the tip for debugging "Connection closed".

In [7]:
import subprocess
p = subprocess.Popen(["mcp", "run", "server.py"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
try:
    p.wait(timeout=5)                      # keep stdin open (closing it = EOF = normal shutdown)
    print("Server exited early! stderr:\n", p.stderr.read())
except subprocess.TimeoutExpired:
    print("✅ Server started and is waiting for an MCP client on STDIO (no startup error).")
    p.kill()

✅ Server started and is waiting for an MCP client on STDIO (no startup error).


### Bonus: talking raw JSON-RPC to the server
To see what the SDK does under the hood, we send the protocol messages ourselves over the server's stdin and read the replies from its stdout.

In [8]:
import json
msgs = [
    {"jsonrpc": "2.0", "id": 1, "method": "initialize",
     "params": {"protocolVersion": "2024-11-05", "capabilities": {}, "clientInfo": {"name": "raw-client", "version": "0.1"}}},
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 20, "b": 22}}},
    {"jsonrpc": "2.0", "id": 3, "method": "resources/read", "params": {"uri": "greeting://Raphael"}},
]
p = subprocess.Popen(["python", "server.py"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
for m in msgs:
    p.stdin.write(json.dumps(m) + "\n"); p.stdin.flush()
    if "id" in m:
        reply = json.loads(p.stdout.readline())
        print(f"--> {m['method']}\n<-- {json.dumps(reply.get('result', reply))[:300]}\n")
p.kill()

--> initialize
<-- {"protocolVersion": "2024-11-05", "capabilities": {"experimental": {}, "prompts": {"listChanged": false}, "resources": {"subscribe": false, "listChanged": false}, "tools": {"listChanged": false}}, "serverInfo": {"name": "Demo", "version": "1.30.0"}}

--> tools/call
<-- {"content": [{"type": "text", "text": "42"}], "structuredContent": {"result": 42}, "isError": false}

--> resources/read
<-- {"contents": [{"uri": "greeting://Raphael", "mimeType": "text/plain", "text": "Hello, Raphael!"}]}



## What to submit
- `server.py` and `client.py` (created above; also saved next to this notebook in the repository);
- the terminal capture (`terminal_capture.txt`), showing:
  - **resource list & tool list:** `Resource templates: ['greeting://{name}']`, `Tools: ['add']`;
  - **`greeting://hello` output:** `Hello, hello!`;
  - **`add(1, 7)` result:** `8` (also returned as structured content `{'result': 8}`).

### Troubleshooting notes (encountered while doing this exercise)
- **`Connection closed`** from the client was caused by the server crashing at import. `pip install "mcp[cli]"` now installs **mcp 2.x**, where `mcp.server.fastmcp` no longer exists (FastMCP became `MCPServer` in `mcp.server.mcpserver`). Running `mcp run server.py` on its own showed the real error. The fix is to pin `mcp[cli]<2` (we used 1.30), or to migrate the code to the v2 API.
- No resources in `list_resources()`: a URI with `{placeholders}` is a **template** and is listed by `list_resource_templates()`.
- `mcp: command not found` → activate the venv where `mcp[cli]` is installed.